# SSN Builder — sequence-based (mmseqs)

Builds **Sequence Similarity Networks (SSN)** from a FASTA file using mmseqs2.  
For structure-based networks (foldseek / PDB), see `build_stsn.ipynb`.

Run cells top to bottom, skipping optional ones as needed.

## Setup

In [3]:
import sys, logging
from pathlib import Path

SRC_DIR = Path("../src").resolve()
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

logging.basicConfig(level=logging.INFO, format="%(asctime)s  %(levelname)-8s  %(message)s", datefmt="%H:%M:%S")
log = logging.getLogger("ssn")

In [4]:
# Shared paths — edit these before running anything else
INPUT_FASTA = Path("/Users/maartenboneschansker/Documents/PhD/collabs/ibaj/supersensitive_sequences/sensitive_GH43.faa")
PREFIX      = "test_run"
OUTPUT_DIR  = Path("/Users/maartenboneschansker/Documents/PhD/collabs/ibaj/supersensitive_sequences/ssn/")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

---
## Step 1 — Redundancy reduction *(optional)*

Run **one** of the two cells below.

### 1a · Cluster with mmseqs

In [3]:
from mmseqs import easy_cluster

result = easy_cluster(
    fasta      = INPUT_FASTA,
    output_dir = OUTPUT_DIR / "cluster",
    prefix     = PREFIX,
    min_seq_id = 0.8,
    coverage   = 0.8,
    cov_mode   = 0,    # 0 = bidirectional, 1 = query, 2 = target
    log        = log,
)
rep_fasta   = result["rep_seq_fasta"]
cluster_tsv = result["cluster_tsv"]
print(f"rep_fasta:   {rep_fasta}")
print(f"cluster_tsv: {cluster_tsv}")

11:52:20  INFO      CMD  mmseqs easy-cluster /Users/maartenboneschansker/Documents/PhD/collabs/ibaj/supersensitive_sequences/sensitive_GH43.faa /Users/maartenboneschansker/Documents/PhD/collabs/ibaj/supersensitive_sequences/ssn/cluster/test_run /Users/maartenboneschansker/Documents/PhD/collabs/ibaj/supersensitive_sequences/ssn/cluster/tmp --min-seq-id 0.8 --cov-mode 0 -c 0.8
11:52:48  INFO           Create directory /Users/maartenboneschansker/Documents/PhD/collabs/ibaj/supersensitive_sequences/ssn/cluster/tmp
11:52:48  INFO           easy-cluster /Users/maartenboneschansker/Documents/PhD/collabs/ibaj/supersensitive_sequences/sensitive_GH43.faa /Users/maartenboneschansker/Documents/PhD/collabs/ibaj/supersensitive_sequences/ssn/cluster/test_run /Users/maartenboneschansker/Documents/PhD/collabs/ibaj/supersensitive_sequences/ssn/cluster/tmp --min-seq-id 0.8 --cov-mode 0 -c 0.8 
11:52:48  INFO           
11:52:48  INFO           MMseqs Version:                     	17.b804f
11:52:48  INFO 

rep_fasta:   /Users/maartenboneschansker/Documents/PhD/collabs/ibaj/supersensitive_sequences/ssn/cluster/test_run_rep_seq.fasta
cluster_tsv: /Users/maartenboneschansker/Documents/PhD/collabs/ibaj/supersensitive_sequences/ssn/cluster/test_run_cluster.tsv


### 1b · Skip clustering — use full input FASTA

In [ ]:
rep_fasta   = INPUT_FASTA
cluster_tsv = None
print(f"Using full input FASTA: {rep_fasta}")

---
## Step 2 — All-vs-all sequence similarity search (mmseqs)

Run **one** of the two cells below.

### 2a · Run mmseqs easy-search

In [4]:
from mmseqs import easy_search

result = easy_search(
    fasta       = rep_fasta,
    output_dir  = OUTPUT_DIR / "search",
    prefix      = PREFIX,
    evalue      = 1e-5,
    sensitivity = 7.5,
    log         = log,
)
m8_path = result["m8"]
print(f"m8: {m8_path}")

11:53:37  INFO      CMD  mmseqs easy-search /Users/maartenboneschansker/Documents/PhD/collabs/ibaj/supersensitive_sequences/ssn/cluster/test_run_rep_seq.fasta /Users/maartenboneschansker/Documents/PhD/collabs/ibaj/supersensitive_sequences/ssn/cluster/test_run_rep_seq.fasta /Users/maartenboneschansker/Documents/PhD/collabs/ibaj/supersensitive_sequences/ssn/search/test_run.m8 /Users/maartenboneschansker/Documents/PhD/collabs/ibaj/supersensitive_sequences/ssn/search/tmp --format-output query,target,pident,alnlen,mismatch,gapopen,qstart,qend,tstart,tend,evalue,bits -e 1e-05 -s 7.5
12:00:26  INFO           Create directory /Users/maartenboneschansker/Documents/PhD/collabs/ibaj/supersensitive_sequences/ssn/search/tmp
12:00:26  INFO           easy-search /Users/maartenboneschansker/Documents/PhD/collabs/ibaj/supersensitive_sequences/ssn/cluster/test_run_rep_seq.fasta /Users/maartenboneschansker/Documents/PhD/collabs/ibaj/supersensitive_sequences/ssn/cluster/test_run_rep_seq.fasta /Users/maart

m8: /Users/maartenboneschansker/Documents/PhD/collabs/ibaj/supersensitive_sequences/ssn/search/test_run.m8


### 2b · Use existing .m8 file

In [6]:
m8_path = Path("/Users/maartenboneschansker/Documents/PhD/collabs/ibaj/supersensitive_sequences/ssn/search/test_run.m8")

---
## Load similarity data

In [7]:
from plot_ssn import load_data

df = load_data(m8_path)
df.head()

14:04:22  INFO      START load similarity table /Users/maartenboneschansker/Documents/PhD/collabs/ibaj/supersensitive_sequences/ssn/search/test_run.m8
14:04:23  INFO      END   load similarity table /Users/maartenboneschansker/Documents/PhD/collabs/ibaj/supersensitive_sequences/ssn/search/test_run.m8: 1.13s
14:04:23  INFO      Loaded 3331909 rows from /Users/maartenboneschansker/Documents/PhD/collabs/ibaj/supersensitive_sequences/ssn/search/test_run.m8
14:04:23  INFO      START remove self-hits
14:04:23  INFO      END   remove self-hits: 0.23s
14:04:23  INFO      Removed self-hits where qseqid == sseqid: 11398 rows
14:04:23  INFO      After removing self-hits: 3320511 rows


Loaded data: 3320511 edges


,qseqid,sseqid,pident,length,mismatch,gapopen,qstart,qend,sstart,send,evalue,bitscore
2,NCC5025_ctg.s1.000000F_882,WFB80974.1,65.9,1067,351,0,1,1030,1,1067,0.000000e+00,1355
3,NCC5025_ctg.s1.000000F_882,WEV53672.1,59.2,1138,446,0,1,1095,1,1138,0.000000e+00,1252
4,NCC5025_ctg.s1.000000F_882,QAY32212.1,60.7,1060,397,0,17,1027,8,1067,0.000000e+00,1221
5,NCC5025_ctg.s1.000000F_882,AEI13085.1,36.5,974,592,0,70,1043,30,962,4.378000e-152,515
6,NCC5025_ctg.s1.000000F_882,AEE44928.1,38.2,931,548,0,97,1027,57,944,4.727000e-146,497


---
## Load annotations *(optional)*

Skip this cell to produce uncolored networks.

In [8]:
import pandas as pd
from plot_ssn import build_color_lookup, _read_meta

META_FILE  = Path("../data/GH43_full.tsv")
ID_COL     = "GenBank"          # column matching sequence IDs in the m8 file
COLOR_COLS = ["Family", "Domain"]     # one plot is produced per column

meta = _read_meta(META_FILE, ID_COL)
print(f"Metadata: {len(meta)} rows  |  columns: {list(meta.columns)}")

color_lookups = {}
for col in COLOR_COLS:
    color_lookups[col] = build_color_lookup(meta, ID_COL, col, clusters=None)
    print(f"  '{col}': {len(color_lookups[col])} annotated entries")

Metadata: 63591 rows  |  columns: ['Family', 'Domain', 'Species', 'GenBank', 'Source']
  'Family': 61969 annotated entries
  'Domain': 61969 annotated entries


---
## Step 3a — Sequence Similarity Network (SSN)

In [1]:
# IDs to highlight with a red circle in the SSN plot.
# Add/remove IDs as needed; set to [] to disable highlighting.
HIGHLIGHT_IDS = [
    "NCC5025_ctg.s1.000000F_882",
    "R11_18m_3_chromosome_248",
]

In [10]:
import json
from plot_ssn import create_graph, exclude_singleton_components, get_layout, plot_ssn, graph_stats

THRESHOLDS         = [0.8]       # pident thresholds (0–1)
NODE_SIZE          = 5
LAYOUT             = "component_grid" # "component_grid" | "fr" | "lgl" | "kk"
EXCLUDE_SINGLETONS = False

# Use color_lookups from the annotations cell, or {} for uncolored plots
_lookups    = locals().get("color_lookups", {})
_meta       = locals().get("meta", None)
_id_col     = locals().get("ID_COL", None)
_color_cols = list(_lookups.keys()) or [None]
_highlight  = locals().get("HIGHLIGHT_IDS", [])

(OUTPUT_DIR / "plots").mkdir(parents=True, exist_ok=True)
ssn_stats = []

for thresh in THRESHOLDS:
    thresh_pident = thresh * 100 if thresh < 1 else thresh
    df_thresh = df[(df["pident"] >= thresh_pident) & (df["qseqid"] != df["sseqid"])].copy()

    g = create_graph(df_thresh)
    if g is None or g.vcount() == 0:
        print(f"t={thresh}: no nodes, skipping."); continue

    if EXCLUDE_SINGLETONS:
        g, _ = exclude_singleton_components(g)
        if g is None or g.vcount() == 0:
            print(f"t={thresh}: empty after singleton removal, skipping."); continue

    print(f"t={thresh}: {g.vcount()} nodes, {g.ecount()} edges")
    layout_coords = get_layout(g, LAYOUT)
    ssn_stats.append(graph_stats(g, thresh, metadata=_meta, id_col=_id_col))

    for col in _color_cols:
        lookup   = _lookups.get(col) or {n: "Unknown" for n in g.vs["name"]}
        col_lbl  = col if col else "uncolored"
        out_file = OUTPUT_DIR / "plots" / f"{PREFIX}_ssn_t{thresh}_{col_lbl}.html"
        plot_ssn(g, str(out_file), col, thresh, str(m8_path), lookup, NODE_SIZE,
                 layout_coords=layout_coords, metadata=_meta, id_col=_id_col,
                 highlight_ids=_highlight)

stats_path = OUTPUT_DIR / "plots" / f"{PREFIX}_ssn_stats.json"
stats_path.write_text(json.dumps(ssn_stats, indent=2))
print(f"\nStats → {stats_path}")
pd.DataFrame(ssn_stats)

14:09:27  INFO      START deduplicate undirected edges
14:09:27  INFO      Deduplicated reciprocal/parallel edges: 4006 -> 2263
14:09:27  INFO      END   deduplicate undirected edges: 0.00s
14:09:27  INFO      START create edge list
14:09:27  INFO      END   create edge list: 0.00s
14:09:27  INFO      START build igraph
14:09:27  INFO      END   build igraph: 0.04s
14:09:27  INFO      START attach edge pident weights
14:09:27  INFO      END   attach edge pident weights: 0.00s
14:09:27  INFO      START compute component_grid layout (2311 nodes, 2263 edges)
14:09:27  INFO      END   compute component_grid layout (2311 nodes, 2263 edges): 0.02s
14:09:27  INFO      START prepare node colors for Family
14:09:27  INFO      END   prepare node colors for Family: 0.00s
14:09:27  INFO      START build edge trace for Family
14:09:27  INFO      END   build edge trace for Family: 0.01s
14:09:27  INFO      START build node traces for Family (discrete)


t=0.8: 2311 nodes, 2263 edges


14:09:31  INFO      END   build node traces for Family (discrete): 4.68s
14:09:31  INFO      START assemble figure for Family
14:09:31  INFO      END   assemble figure for Family: 0.03s
14:09:31  INFO      START write HTML /Users/maartenboneschansker/Documents/PhD/collabs/ibaj/supersensitive_sequences/ssn/plots/test_run_ssn_t0.8_Family.html
14:09:31  INFO      END   write HTML /Users/maartenboneschansker/Documents/PhD/collabs/ibaj/supersensitive_sequences/ssn/plots/test_run_ssn_t0.8_Family.html: 0.03s
14:09:31  INFO      Saved: /Users/maartenboneschansker/Documents/PhD/collabs/ibaj/supersensitive_sequences/ssn/plots/test_run_ssn_t0.8_Family.html
14:09:31  INFO      START write PNG /Users/maartenboneschansker/Documents/PhD/collabs/ibaj/supersensitive_sequences/ssn/plots/test_run_ssn_t0.8_Family.png


  Saved: /Users/maartenboneschansker/Documents/PhD/collabs/ibaj/supersensitive_sequences/ssn/plots/test_run_ssn_t0.8_Family.html


14:09:32  INFO      END   write PNG /Users/maartenboneschansker/Documents/PhD/collabs/ibaj/supersensitive_sequences/ssn/plots/test_run_ssn_t0.8_Family.png: 0.39s
14:09:32  INFO      Saved: /Users/maartenboneschansker/Documents/PhD/collabs/ibaj/supersensitive_sequences/ssn/plots/test_run_ssn_t0.8_Family.png
14:09:32  INFO      START prepare node colors for Domain
14:09:32  INFO      END   prepare node colors for Domain: 0.00s
14:09:32  INFO      START build edge trace for Domain
14:09:32  INFO      END   build edge trace for Domain: 0.02s
14:09:32  INFO      START build node traces for Domain (discrete)


  Saved: /Users/maartenboneschansker/Documents/PhD/collabs/ibaj/supersensitive_sequences/ssn/plots/test_run_ssn_t0.8_Family.png


14:09:37  INFO      END   build node traces for Domain (discrete): 4.78s
14:09:37  INFO      START assemble figure for Domain
14:09:37  INFO      END   assemble figure for Domain: 0.02s
14:09:37  INFO      START write HTML /Users/maartenboneschansker/Documents/PhD/collabs/ibaj/supersensitive_sequences/ssn/plots/test_run_ssn_t0.8_Domain.html
14:09:37  INFO      END   write HTML /Users/maartenboneschansker/Documents/PhD/collabs/ibaj/supersensitive_sequences/ssn/plots/test_run_ssn_t0.8_Domain.html: 0.02s
14:09:37  INFO      Saved: /Users/maartenboneschansker/Documents/PhD/collabs/ibaj/supersensitive_sequences/ssn/plots/test_run_ssn_t0.8_Domain.html
14:09:37  INFO      START write PNG /Users/maartenboneschansker/Documents/PhD/collabs/ibaj/supersensitive_sequences/ssn/plots/test_run_ssn_t0.8_Domain.png


  Saved: /Users/maartenboneschansker/Documents/PhD/collabs/ibaj/supersensitive_sequences/ssn/plots/test_run_ssn_t0.8_Domain.html


14:09:37  INFO      END   write PNG /Users/maartenboneschansker/Documents/PhD/collabs/ibaj/supersensitive_sequences/ssn/plots/test_run_ssn_t0.8_Domain.png: 0.31s
14:09:37  INFO      Saved: /Users/maartenboneschansker/Documents/PhD/collabs/ibaj/supersensitive_sequences/ssn/plots/test_run_ssn_t0.8_Domain.png


  Saved: /Users/maartenboneschansker/Documents/PhD/collabs/ibaj/supersensitive_sequences/ssn/plots/test_run_ssn_t0.8_Domain.png

Stats → /Users/maartenboneschansker/Documents/PhD/collabs/ibaj/supersensitive_sequences/ssn/plots/test_run_ssn_stats.json


,threshold,nodes,edges,n_components,largest_component,singletons,annotated_nodes,annotated_percent
0,0.8,2311,2263,595,50,0,2310,100.0


---
## Step 3b — Minimum Spanning Tree / StSN *(optional)*

Reduces each connected component to its spanning tree (highest-similarity edges only).  
Useful for structure-based networks (foldseek) or for cleaner visualizations of dense SSNs.

In [12]:
from plot_mst import (
    build_mst_graph,
    exclude_singleton_components as mst_exclude_singletons,
    get_layout as mst_get_layout,
    plot_mst,
    graph_stats as mst_graph_stats,
)

THRESHOLDS         = [0.4, 0.6]       # pident thresholds (0–1)
NODE_SIZE          = 2
LAYOUT             = "component_grid"
EXCLUDE_SINGLETONS = True

_lookups    = locals().get("color_lookups", {})
_meta       = locals().get("meta", None)
_id_col     = locals().get("ID_COL", None)
_color_cols = list(_lookups.keys()) or [None]

(OUTPUT_DIR / "plots").mkdir(parents=True, exist_ok=True)
mst_stats = []

for thresh in THRESHOLDS:
    thresh_pident = thresh * 100 if thresh < 1 else thresh
    df_thresh = df[(df["pident"] >= thresh_pident) & (df["qseqid"] != df["sseqid"])].copy()

    g = build_mst_graph(df_thresh)
    if g is None or g.vcount() == 0:
        print(f"t={thresh}: no nodes, skipping."); continue

    if EXCLUDE_SINGLETONS:
        g, _ = mst_exclude_singletons(g)
        if g is None or g.vcount() == 0:
            print(f"t={thresh}: empty after singleton removal, skipping."); continue

    print(f"t={thresh}: {g.vcount()} nodes, {g.ecount()} MST edges")
    layout_coords = mst_get_layout(g, LAYOUT)
    mst_stats.append(mst_graph_stats(g, thresh, metadata=_meta, id_col=_id_col))

    for col in _color_cols:
        lookup   = _lookups.get(col) or {n: "Unknown" for n in g.vs["name"]}
        col_lbl  = col if col else "uncolored"
        out_file = OUTPUT_DIR / "plots" / f"{PREFIX}_mst_t{thresh}_{col_lbl}.html"
        plot_mst(g, str(out_file), col, thresh, str(m8_path), lookup, NODE_SIZE,
                 layout_coords=layout_coords, metadata=_meta, id_col=_id_col)

stats_path = OUTPUT_DIR / "plots" / f"{PREFIX}_mst_stats.json"
stats_path.write_text(json.dumps(mst_stats, indent=2))
print(f"\nStats → {stats_path}")
pd.DataFrame(mst_stats)

13:46:14  INFO      START deduplicate undirected edges
13:46:14  INFO      Deduplicated edges: 86335 -> 45949
13:46:14  INFO      END   deduplicate undirected edges: 0.03s
13:46:14  INFO      START build node index
13:46:14  INFO      END   build node index: 0.00s
13:46:14  INFO      START build sparse distance matrix
13:46:14  INFO      END   build sparse distance matrix: 0.01s
13:46:14  INFO      START compute minimum spanning tree (forest)
13:46:14  INFO      END   compute minimum spanning tree (forest): 0.00s
13:46:14  INFO      START extract MST edges
13:46:14  INFO      MST has 2458 edges (from 45949 input edges)
13:46:14  INFO      END   extract MST edges: 0.00s
13:46:14  INFO      START build igraph from MST edges
13:46:14  INFO      END   build igraph from MST edges: 0.00s
13:46:14  INFO      START compute component_grid layout (2480 nodes, 2458 edges)
13:46:14  INFO      END   compute component_grid layout (2480 nodes, 2458 edges): 0.85s


t=0.4: 2480 nodes, 2458 MST edges


13:46:14  INFO      START prepare node colors for Family
13:46:14  INFO      END   prepare node colors for Family: 0.00s
13:46:14  INFO      START build edge trace for Family
13:46:14  INFO      END   build edge trace for Family: 0.02s
13:46:14  INFO      START build node traces for Family (discrete)
13:46:19  INFO      END   build node traces for Family (discrete): 4.96s
13:46:19  INFO      START assemble figure for Family
13:46:19  INFO      END   assemble figure for Family: 0.03s
13:46:19  INFO      START write HTML ../results/GH43_test/plots/test_run_mst_t0.4_Family.html
13:46:19  INFO      END   write HTML ../results/GH43_test/plots/test_run_mst_t0.4_Family.html: 0.02s
13:46:19  INFO      Saved: ../results/GH43_test/plots/test_run_mst_t0.4_Family.html
13:46:19  INFO      START write PNG ../results/GH43_test/plots/test_run_mst_t0.4_Family.png


  Saved: ../results/GH43_test/plots/test_run_mst_t0.4_Family.html


13:46:20  INFO      END   write PNG ../results/GH43_test/plots/test_run_mst_t0.4_Family.png: 0.37s
13:46:20  INFO      Saved: ../results/GH43_test/plots/test_run_mst_t0.4_Family.png
13:46:20  INFO      START prepare node colors for Domain
13:46:20  INFO      END   prepare node colors for Domain: 0.00s
13:46:20  INFO      START build edge trace for Domain
13:46:20  INFO      END   build edge trace for Domain: 0.02s
13:46:20  INFO      START build node traces for Domain (discrete)


  Saved: ../results/GH43_test/plots/test_run_mst_t0.4_Family.png


... (7 outputs left out)

  Saved: ../results/GH43_test/plots/test_run_mst_t0.6_Family.png


13:46:30  INFO      END   build node traces for Domain (discrete): 2.36s
13:46:30  INFO      START assemble figure for Domain
13:46:30  INFO      END   assemble figure for Domain: 0.01s
13:46:30  INFO      START write HTML ../results/GH43_test/plots/test_run_mst_t0.6_Domain.html
13:46:30  INFO      END   write HTML ../results/GH43_test/plots/test_run_mst_t0.6_Domain.html: 0.02s
13:46:30  INFO      Saved: ../results/GH43_test/plots/test_run_mst_t0.6_Domain.html
13:46:30  INFO      START write PNG ../results/GH43_test/plots/test_run_mst_t0.6_Domain.png
13:46:30  INFO      END   write PNG ../results/GH43_test/plots/test_run_mst_t0.6_Domain.png: 0.17s
13:46:30  INFO      Saved: ../results/GH43_test/plots/test_run_mst_t0.6_Domain.png


  Saved: ../results/GH43_test/plots/test_run_mst_t0.6_Domain.html
  Saved: ../results/GH43_test/plots/test_run_mst_t0.6_Domain.png

Stats → ../results/GH43_test/plots/test_run_mst_stats.json


,threshold,nodes,mst_edges,n_components,largest_component,singletons,annotated_nodes,annotated_percent
0,0.4,2480,2458,22,2369,0,2480,100.0
1,0.6,1205,1078,127,302,0,1205,100.0


---
## Results

In [ ]:
from IPython.display import display, HTML

html_files = sorted((OUTPUT_DIR / "plots").glob("*.html"))
items = "".join(f'<li><a href="{f.resolve()}" target="_blank">{f.name}</a></li>' for f in html_files)
display(HTML(f"<ul>{items}</ul>") if items else HTML("<i>No plots found — run cells above first.</i>"))